In [3]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
df = pd.read_excel(r"C:\Users\ODAMA\Downloads\05_Jumia_Customer_Analysis.xlsx",
                   sheet_name='Customer Data',
                   header=2)

# ── KPIs ──────────────────────────────────────────────────────────────────────
total_customers  = len(df)
total_revenue    = df['Total Spent (₦)'].sum()
avg_loyalty      = df['Loyalty Score'].mean()
avg_refund       = df['Refund Rate %'].mean()

# ── CHART DATA ────────────────────────────────────────────────────────────────

# 1. Customer Status breakdown
status_count = df.groupby('Customer Status')['Customer ID'].count().reset_index()
status_count.columns = ['Customer Status', 'Count']

# 2. Segment distribution
segment_count = df.groupby('Segment')['Customer ID'].count().reset_index()
segment_count.columns = ['Segment', 'Count']

# 3. Total Spent by State
spent_by_state = df.groupby('State')['Total Spent (₦)'].sum().reset_index().sort_values('Total Spent (₦)')

# 4. Device Used distribution
device_count = df.groupby('Device Used')['Customer ID'].count().reset_index()
device_count.columns = ['Device Used', 'Count']

# 5. Scatter — Loyalty Score vs Total Spent
# used directly from df

# 6. Top 10 Customers by Total Spent
top_customers = df.nlargest(10, 'Total Spent (₦)')[['Full Name','Total Spent (₦)','Segment','Customer Status']].sort_values('Total Spent (₦)')

# ── CHARTS ────────────────────────────────────────────────────────────────────

# Chart style helper
def style(fig):
    fig.update_layout(
        plot_bgcolor='#ffffff',
        paper_bgcolor='#ffffff',
        font=dict(color='#1f3b5c', family='Arial', size=13),
        title_font=dict(size=16, color='#1f3b5c', family='Arial'),
        margin=dict(l=20, r=20, t=50, b=100),  # 👈 increased bottom margin
        height=400,                              # 👈 slightly taller
        showlegend=True,
        legend=dict(
            bgcolor='rgba(255,255,255,0.95)',
            bordercolor='#d6e6f2',
            borderwidth=1,
            font=dict(size=12, color='#1f3b5c'),
            orientation='h',       # horizontal legend
            yanchor='top',         # 👈 changed
            y=-0.25,               # 👈 moves legend below the chart
            xanchor='center',      # 👈 centers it
            x=0.5                  # 👈 centered
        ),
        xaxis=dict(tickangle=45, tickfont=dict(size=11), title_font=dict(size=13),
                   showgrid=True, gridcolor='#f0f4f8'),
        yaxis=dict(tickfont=dict(size=11), title_font=dict(size=13),
                   showgrid=True, gridcolor='#f0f4f8')
    )
    return fig

# 1. Customer Status — donut
fig_status = px.pie(status_count, names='Customer Status', values='Count',
                    hole=0.5, title='Customer Status Breakdown',
                    color='Customer Status',
                    color_discrete_map={
                        'Active':  '#1A7A4A',
                        'At Risk': '#E67E22',
                        'Churned': '#C0392B'
                    })
style(fig_status)

# 2. Segment distribution — bar
fig_segment = px.bar(segment_count, x='Segment', y='Count',
                     color='Segment',
                     color_discrete_sequence=['#08306b','#2171b5','#6baed6','#c6dbef'],
                     title='Customers by Segment')
style(fig_segment)

# 3. Total Spent by State — horizontal bar
fig_state = px.bar(spent_by_state, x='Total Spent (₦)', y='State',
                   orientation='h', color='Total Spent (₦)',
                   color_continuous_scale='Blues',
                   title='Total Spent by State')
style(fig_state)

# 4. Device Used — donut
fig_device = px.pie(device_count, names='Device Used', values='Count',
                    hole=0.5, title='Device Used by Customers',
                    color_discrete_sequence=['#08306b','#2171b5','#6baed6'])
style(fig_device)

# 5. Scatter — Loyalty Score vs Total Spent
fig_scatter = px.scatter(df, x='Loyalty Score', y='Total Spent (₦)',
                         color='Customer Status',
                         size='Total Orders',
                         hover_data=['Full Name', 'Segment', 'State'],
                         title='Loyalty Score vs Total Spent',
                         color_discrete_map={
                             'Active':  '#1A7A4A',
                             'At Risk': '#E67E22',
                             'Churned': '#C0392B'
                         })
fig_scatter.update_traces(marker=dict(
    line=dict(width=2, color='black'),
    opacity=0.85,
    size=10
))
style(fig_scatter)

# 6. Top 10 Customers — horizontal bar
fig_top = px.bar(top_customers, x='Total Spent (₦)', y='Full Name',
                 orientation='h', color='Customer Status',
                 color_discrete_map={
                     'Active':  '#1A7A4A',
                     'At Risk': '#E67E22',
                     'Churned': '#C0392B'
                 },
                 title='Top 10 Customers by Total Spent')
style(fig_top)

# ── STYLES ────────────────────────────────────────────────────────────────────
BLUE_DARK  = '#1f3b5c'
BLUE_MID   = '#2171b5'
BLUE_LIGHT = '#d6e6f2'
WHITE      = '#ffffff'

kpi_card = {
    'backgroundColor': WHITE,
    'padding': '16px 22px',
    'borderRadius': '12px',
    'textAlign': 'center',
    'flex': '1',
    'margin': '6px',
    'border': f'1.5px solid {BLUE_LIGHT}',
    'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)'
}
chart_box = {
    'backgroundColor': WHITE,
    'padding': '10px',
    'borderRadius': '12px',
    'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)',
    'flex': '1',
    'border': f'1px solid {BLUE_LIGHT}'
}
sidebar_label = {
    'backgroundColor': BLUE_MID,
    'color': WHITE,
    'padding': '7px 12px',
    'borderRadius': '6px',
    'marginBottom': '7px',
    'fontSize': '13px',
    'fontWeight': 'bold',
    'textAlign': 'center'
}

# ── APP LAYOUT ────────────────────────────────────────────────────────────────
app = Dash(__name__)

app.layout = html.Div(
    style={'backgroundColor': '#eaf3f9', 'padding': '20px', 'fontFamily': 'Arial'},
    children=[

        # Title + KPI Row
        html.Div([
            html.Div(
                html.H2('JUMIA NIGERIA — CUSTOMER ANALYSIS DASHBOARD',
                        style={'color': WHITE, 'margin': '0', 'fontSize': '20px'}),
                style={'backgroundColor': BLUE_DARK, 'padding': '16px 22px',
                       'borderRadius': '12px', 'flex': '2', 'marginRight': '12px'}
            ),
            html.Div([
                html.Div([
                    html.P('TOTAL CUSTOMERS', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'{total_customers:,}', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('TOTAL REVENUE', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'₦{total_revenue/1e6:,.1f}M', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('AVG LOYALTY', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'{avg_loyalty:.1f} / 10', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('AVG REFUND RATE', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'{avg_refund:.1f}%', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
            ], style={'display': 'flex', 'flex': '4'})
        ], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '16px'}),

        # Body
        html.Div([

            # Sidebar
            html.Div([
                html.P('Status', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'8px','fontSize':'13px'}),
                html.Div('Active',  style={**sidebar_label, 'backgroundColor': '#1A7A4A'}),
                html.Div('At Risk', style={**sidebar_label, 'backgroundColor': '#E67E22'}),
                html.Div('Churned', style={**sidebar_label, 'backgroundColor': '#C0392B'}),
                html.Br(),
                html.P('Segment', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'8px','fontSize':'13px'}),
                *[html.Div(s, style=sidebar_label) for s in ['VIP','Premium','Regular','Occasional','New']],
                html.Br(),
                html.P('Device', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'8px','fontSize':'13px'}),
                *[html.Div(d, style=sidebar_label) for d in ['Mobile App','Web Browser','USSD']],
                html.Br(), html.Br(),
                html.P('Prepared by', style={'fontSize':'12px','color':BLUE_DARK,'margin':'0'}),
                html.P('Odama Joseph', style={'fontSize':'13px','color':BLUE_DARK,'fontWeight':'bold','margin':'0'}),
            ], style={
                'width': '160px', 'minWidth': '160px',
                'backgroundColor': WHITE, 'padding': '16px',
                'borderRadius': '12px', 'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)',
                'marginRight': '12px', 'border': f'1px solid {BLUE_LIGHT}'
            }),

            # Charts — 2 per row
            html.Div([
                html.Div([
                    html.Div(dcc.Graph(figure=fig_status,  config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_segment, config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),

                html.Div([
                    html.Div(dcc.Graph(figure=fig_state,   config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_device,  config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),

                html.Div([
                    html.Div(dcc.Graph(figure=fig_scatter, config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_top,     config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px'}),

            ], style={'flex': '1'})

        ], style={'display': 'flex', 'alignItems': 'flex-start'})
    ]
)

# ── RUN ───────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    app.run(debug=True, port=8055)